# Baseline Evaluation — MASSIVE-Agents

**Penelitian:** Evaluasi Agentic AI pada Tugas dan Bahasa Domain Lokal  
**Dataset:** MASSIVE-Agents (en-US dan id-ID)

Pengujian menggunakan tiga model: Qwen2.5-7B, Gemma 2 9B, dan Sahabat-AI 9B. Tahap awal menggunakan sampel development untuk memeriksa pembacaan data, prediksi fungsi dan argumen, serta hasil penilaian. Data test belum digunakan.

## 1. Konfigurasi pengujian

Setiap record diuji dengan 55 fungsi yang tersedia pada dataset. Tugas dan parameter pengujian dibuat sama untuk ketiga model, sedangkan format prompt disesuaikan dengan template model.

Mode awal adalah `pilot`. Pengujian development penuh menggunakan `development_full`. Data test hanya digunakan setelah konfigurasi evaluasi ditetapkan.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import re
import sys
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display

# Model yang diuji
MODELS = [
    "qwen2.5:7b-instruct",
    "gemma2:9b-instruct-q4_K_M",
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
]

API_BASE = "http://127.0.0.1:11434"
RUN_MODE = "pilot"  # pilot | development_full | test_full
PILOT_PAIRS = 2       # Dua ID paralel = empat record; total 12 permintaan untuk 3 model
ALLOW_TEST_EVALUATION = False
RANDOM_SEED = 20261020
TIMEOUT_SECONDS = 1200
OPTIONS = {
    "temperature": 0,
    "seed": RANDOM_SEED,
    "num_ctx": 16384,
    "num_predict": 384,
    "num_batch": 128,
}
KEEP_ALIVE = "10m"
PROMPT_VERSION = "massive_agents_full_catalog_json_v1"

if RUN_MODE not in {"pilot", "development_full", "test_full"}:
    raise ValueError("RUN_MODE harus pilot, development_full, atau test_full")
if RUN_MODE == "test_full" and not ALLOW_TEST_EVALUATION:
    raise RuntimeError("Test set dikunci. Set ALLOW_TEST_EVALUATION=True hanya setelah pilot dan development disetujui.")

print("Mode:", RUN_MODE)
print("Model:", len(MODELS))
print("Konfigurasi:", OPTIONS)
print("Python:", sys.version.split()[0])

Mode: pilot
Model: 3
Konfigurasi: {'temperature': 0, 'seed': 20261020, 'num_ctx': 16384, 'num_predict': 384, 'num_batch': 128}
Python: 3.13.12


## 2. Lokasi dan kondisi dataset

Data diambil dari `data/processed/massive_agents_reviewed`. Jumlah record dan status pembagiannya diperiksa menggunakan `results/preprocessing_reviewed/finalization_summary.json`.

Ground truth (GT) asli tidak diubah. Keputusan audit manual pada dataset ini masih bersifat sementara.

In [2]:
def locate_project():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"D:\\KP-Agentic-AI")]
    for root in candidates:
        if (root / "data" / "processed" / "massive_agents_reviewed").is_dir():
            return root
    raise FileNotFoundError(
        "Folder data/processed/massive_agents_reviewed tidak ditemukan. "
        "Buka notebook dari proyek KP-Agentic-AI di VS Code."
    )

ROOT = locate_project()
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
REPORT_DIR = ROOT / "results" / "preprocessing_reviewed"
OUT = ROOT / "results" / "baseline_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

summary_path = REPORT_DIR / "finalization_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Laporan audit tidak ditemukan: {summary_path}")
review_summary = json.loads(summary_path.read_text(encoding="utf-8-sig"))
assert review_summary.get("anti_leakage_group_and_id") == "PASS", "Laporan anti-leakage belum PASS"
assert review_summary.get("gt_modified") is False, "Laporan menunjukkan GT berubah"

print("Folder proyek:", ROOT)
print("Dataset:", DATA_DIR)
print("Status audit:", review_summary.get("audit_status", "tidak tercatat"))
print("Kandidat evaluasi:", review_summary.get("ready_to_evaluate_candidate"))
print("Development:", review_summary.get("development"))
print("Test:", review_summary.get("test"))

Folder proyek: D:\KP-Agentic-AI
Dataset: D:\KP-Agentic-AI\data\processed\massive_agents_reviewed
Status audit: provisional decisions; methodological/source verification still required
Kandidat evaluasi: 3281
Development: 979
Test: 2302


## 3. Pemeriksaan record

Setiap record diperiksa untuk memastikan identitas, bahasa, pertanyaan, katalog fungsi, dan GT tersedia. Katalog fungsi harus berisi 55 fungsi, termasuk fungsi target dari GT. Data dengan struktur tidak sesuai akan dihentikan untuk diperiksa.

In [3]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    rows = []
    with path.open("r", encoding="utf-8-sig") as fh:
        for line_no, line in enumerate(fh, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            required = {"row_key", "source_record_id", "language_code", "group_id", "split", "question", "function", "gt"}
            missing = required.difference(row)
            if missing:
                raise ValueError(f"{path.name} baris {line_no}: kolom hilang {missing}")
            if not isinstance(row["function"], list) or not row["function"]:
                raise ValueError(f"{row['row_key']}: katalog fungsi kosong")
            if len(row["gt"]) != 1:
                raise ValueError(f"{row['row_key']}: GT memiliki {len(row['gt'])} fungsi, perlu ditinjau")
            names = [f.get("name") for f in row["function"] if isinstance(f, dict)]
            target = next(iter(row["gt"]))
            if target not in names:
                raise ValueError(f"{row['row_key']}: fungsi target tidak ditawarkan")
            if len(set(names)) != len(names):
                raise ValueError(f"{row['row_key']}: nama fungsi duplikat")
            rows.append(row)
    return rows

split_name = "test" if RUN_MODE == "test_full" else "development"
dataset = {
    lang: read_jsonl(DATA_DIR / f"{lang}_{split_name}.jsonl")
    for lang in ("en-US", "id-ID")
}

all_rows = [row for part in dataset.values() for row in part]
assert len({r["row_key"] for r in all_rows}) == len(all_rows), "row_key duplikat"
assert all(r["split"] == split_name for r in all_rows), "Label split tidak sesuai"
assert all(len(r["function"]) == 55 for r in all_rows), "Jumlah katalog fungsi bukan 55; tinjau data terlebih dahulu"
assert all(r["language_code"] in dataset for r in all_rows)

expected_count = review_summary.get(split_name)
if expected_count is not None:
    assert len(all_rows) == expected_count, f"Jumlah {split_name} berbeda dari laporan audit"

print("Split:", split_name)
for language in dataset:
    print(language, len(dataset[language]), "record")
print("Total:", len(all_rows))
print("Fungsi yang tersedia per record: 55")

Split: development
en-US 582 record
id-ID 397 record
Total: 979
Fungsi yang tersedia per record: 55


## 4. Pemilihan data pengujian

Pilot menggunakan dua ID yang tersedia dalam bahasa Inggris dan Indonesia pada development set, sehingga terdapat empat record untuk setiap model. Pemilihan memakai seed tetap agar dapat diulang.

Pasangan pada tahap ini digunakan untuk memeriksa alur evaluasi, bukan untuk menyimpulkan kesetaraan makna EN–ID. Mode `development_full` menggunakan seluruh data development.

In [4]:
def target_name(row):
    return next(iter(row["gt"]))

if RUN_MODE == "pilot":
    index = {
        lang: {str(r["source_record_id"]): r for r in records}
        for lang, records in dataset.items()
    }
    both_ids = sorted(set(index["en-US"]) & set(index["id-ID"]))
    if len(both_ids) < PILOT_PAIRS:
        raise ValueError(f"Pasangan development tersedia hanya {len(both_ids)}; kurangi PILOT_PAIRS")
    randomizer = random.Random(RANDOM_SEED)
    randomizer.shuffle(both_ids)
    chosen, seen_targets = [], set()
    for record_id in both_ids:
        en, ind = index["en-US"][record_id], index["id-ID"][record_id]
        if target_name(en) == target_name(ind) and target_name(en) not in seen_targets:
            chosen.append(record_id)
            seen_targets.add(target_name(en))
        if len(chosen) >= PILOT_PAIRS:
            break
    if len(chosen) < PILOT_PAIRS:
        for record_id in both_ids:
            if record_id not in chosen:
                chosen.append(record_id)
            if len(chosen) >= PILOT_PAIRS:
                break
    selected = [index[lang][record_id] for record_id in chosen for lang in ("en-US", "id-ID")]
else:
    selected = sorted(all_rows, key=lambda r: (str(r["source_record_id"]), r["language_code"]))

assert len({r["row_key"] for r in selected}) == len(selected)
print("Jumlah record dipilih:", len(selected))
print("Jumlah total permintaan:", len(selected) * len(MODELS))
for r in selected[:8]:
    print(r["source_record_id"], r["language_code"], target_name(r), "|", r["question"][:85])

Jumlah record dipilih: 4
Jumlah total permintaan: 12
4168 en-US weather.query | please check the weather in kansas
4168 id-ID weather.query | tolong cek cuaca di kendari
15794 en-US email.querycontact | where seth lives
15794 id-ID email.querycontact | dimana siska tinggal


## 5. Penyusunan input model

Input terdiri dari pertanyaan asli dan katalog 55 fungsi pada record. GT tidak disertakan dalam prompt. Model diminta mengembalikan JSON dengan `name` (fungsi) dan `arguments` (argumen).

Qwen2.5 dan Gemma 2 menggunakan API chat Ollama. Sahabat-AI menggunakan format prompt Gemma 2 secara langsung karena template bawaan model yang dipasang menghasilkan respons tidak sesuai pada pemeriksaan awal. Cara pengiriman ini dicatat sebagai perbedaan konfigurasi.

In [5]:
INSTRUCTION = (
    "Select exactly one function for the user request using the available function catalog. "
    "Do not call the function or answer the question. "
    "Return exactly one JSON object with keys 'name' and 'arguments'. "
    "Use the exact function name and argument names from the catalog. "
    "For each argument, output a string value. Omit arguments that are not specified. "
    "Do not invent values, translate proper names, or include explanations. "
    "Your response must be valid JSON."
)

def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

def task_content(record):
    # Data fungsi tetap sama; format JSON dipadatkan.
    catalog = compact_json(record["function"])
    return (
        INSTRUCTION
        + "\n\nAvailable function catalog (JSON):\n"
        + catalog
        + "\n\nUser request:\n"
        + record["question"]
        + "\n\nOutput JSON only:"
    )

def build_request(model, record):
    content = task_content(record)
    common = {
        "model": model, "stream": False,
        "options": dict(OPTIONS), "keep_alive": KEEP_ALIVE,
    }
    if model.startswith("Supa-AI/"):
        prompt = "<start_of_turn>user\n" + content + "\n<end_of_turn>\n<start_of_turn>model\n"
        common.update({
            "prompt": prompt, "raw": True,
            "options": {**OPTIONS, "stop": ["<end_of_turn>", "<eos>"]},
        })
        return "/api/generate", common
    common.update({
        "messages": [{"role": "user", "content": content}],
        "format": "json",
    })
    return "/api/chat", common

sample_text = task_content(selected[0])
print("Contoh input:", selected[0]["row_key"])
print("Ukuran katalog dan instruksi:", len(sample_text), "karakter")
print("Jumlah fungsi:", len(selected[0]["function"]))
print("Tidak ada GT di input:", "ground_truth" not in sample_text)
print("Panjang input relatif besar; pemrosesan bisa lambat pada GPU 8 GB.")

Contoh input: en-US:0000529
Ukuran katalog dan instruksi: 40188 karakter
Jumlah fungsi: 55
Tidak ada GT di input: True
Panjang input relatif besar; pemrosesan bisa lambat pada GPU 8 GB.


## 6. Penilaian prediksi

Penilaian memeriksa format JSON, pilihan fungsi, nama argumen, dan nilai argumen. Nilai GT yang memiliki beberapa alternatif diperiksa terhadap daftar nilai yang tersedia. Argumen dengan alternatif kosong dapat tidak disertakan dalam prediksi.

Status hasil meliputi `correct`, `wrong_function`, `wrong_argument_keys`, `wrong_argument_values`, `invalid_json`, `invalid_structure`, dan `api_error`.

Perhitungan ini merupakan **scoring internal**, bukan AST Accuracy resmi pada paper MASSIVE-Agents. Perbedaan terjemahan nilai argumen juga belum diperiksa secara semantik.

In [6]:
def normalize_value(value):
    return " ".join(value.split()).casefold()

def parse_model_json(raw):
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None, "invalid_json"
    if not isinstance(obj, dict):
        return None, "invalid_structure"
    if not isinstance(obj.get("name"), str) or not isinstance(obj.get("arguments"), dict):
        return None, "invalid_structure"
    return obj, "parsed"

def score_internal(prediction, parsing, gt):
    if parsing != "parsed":
        return parsing, False, False
    function_name, gold_args = next(iter(gt.items()))
    correct_fn = prediction["name"] == function_name
    if not correct_fn:
        return "wrong_function", False, False

    predicted_args = prediction["arguments"]
    gold_keys = set(gold_args)
    if set(predicted_args) - gold_keys:
        return "wrong_argument_keys", True, False

    # Argumen dengan alternatif kosong boleh tidak diisi.
    mandatory = {k for k, alternatives in gold_args.items() if "" not in alternatives}
    if mandatory - set(predicted_args):
        return "wrong_argument_keys", True, False

    for key, value in predicted_args.items():
        allowed = gold_args[key]
        if not isinstance(value, str) or not isinstance(allowed, list):
            return "wrong_argument_values", True, False
        if normalize_value(value) not in {normalize_value(str(item)) for item in allowed}:
            return "wrong_argument_values", True, False
    return "correct", True, True

# Pemeriksaan logika penilaian.
test_gt = {"demo.tool": {"city": ["Jakarta"], "date": ["", "today"]}}
assert score_internal({"name": "demo.tool", "arguments": {"city": "Jakarta"}}, "parsed", test_gt)[0] == "correct"
assert score_internal({"name": "other", "arguments": {"city": "Jakarta"}}, "parsed", test_gt)[0] == "wrong_function"
assert score_internal({"name": "demo.tool", "arguments": {}}, "parsed", test_gt)[0] == "wrong_argument_keys"
assert score_internal({"name": "demo.tool", "arguments": {"city": "Bandung"}}, "parsed", test_gt)[0] == "wrong_argument_values"
print("Pemeriksaan fungsi scoring: LULUS")

Pemeriksaan fungsi scoring: LULUS


## 7. Pemeriksaan Ollama

Koneksi server lokal dan ketersediaan ketiga model diperiksa sebelum pengujian. Identitas model disimpan sebagai bagian dari konfigurasi eksperimen.

In [7]:
def ollama_request(endpoint, payload=None, timeout=TIMEOUT_SECONDS):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        API_BASE + endpoint,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST" if data is not None else "GET",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        msg = exc.read().decode("utf-8", errors="replace")[:800]
        raise RuntimeError(f"HTTP {exc.code}: {msg}") from exc

try:
    tags = ollama_request("/api/tags", timeout=20)
except Exception as exc:
    raise RuntimeError("Ollama belum dapat diakses di 127.0.0.1:11434. Jalankan ollama serve.") from exc

installed = {item.get("name"): item for item in tags.get("models", [])}
missing_models = [name for name in MODELS if name not in installed]
if missing_models:
    raise RuntimeError(f"Model belum tersedia: {missing_models}")

for model in MODELS:
    info = installed[model]
    print(model, "| ID:", info.get("digest", "")[:12], "| byte:", info.get("size"))

qwen2.5:7b-instruct | ID: 845dbda0ea48 | byte: 4683087332
gemma2:9b-instruct-q4_K_M | ID: c20bec88025f | byte: 5761067555
Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m | ID: 6510227e6320 | byte: 5761058948


## 8. Menjalankan pengujian

Ketiga model memproses record yang sama secara bergantian. Setiap hasil langsung disimpan ke CSV. Jika pengujian diulang dengan konfigurasi yang sama, hasil yang sudah tersimpan tidak diproses ulang.

Respons kosong, format tidak valid, dan kegagalan API tetap dicatat sebagai hasil pengujian.

In [8]:
def create_run_fingerprint():
    identity = {
        "prompt_version": PROMPT_VERSION, "models": MODELS,
        "mode": RUN_MODE, "options": OPTIONS, "seed": RANDOM_SEED,
        "row_keys": [r["row_key"] for r in selected],
        "model_digests": {m: installed[m].get("digest") for m in MODELS},
    }
    return hashlib.sha256(compact_json(identity).encode("utf-8")).hexdigest()[:16]

RUN_FINGERPRINT = create_run_fingerprint()
RESULT_FILE = OUT / f"predictions_{RUN_MODE}.csv"
SUMMARY_FILE = OUT / f"summary_{RUN_MODE}.csv"
CONFIG_FILE = OUT / f"config_{RUN_MODE}.json"

RESULT_FIELDS = [
    "fingerprint", "model", "row_key", "source_record_id", "language_code", "group_id", "split",
    "target_function", "question", "prediction_name", "prediction_arguments_json",
    "gt_json", "status", "parsed_json", "function_correct", "full_match",
    "done_reason", "prompt_eval_count", "eval_count", "latency_seconds",
    "raw_output", "error", "timestamp_utc"
]

def read_previous(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))

def save_predictions(rows):
    # Simpan melalui file sementara untuk mencegah CSV terpotong.
    temp = RESULT_FILE.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=RESULT_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(RESULT_FILE)

previous = read_previous(RESULT_FILE)
if previous and any(row.get("fingerprint") != RUN_FINGERPRINT for row in previous):
    raise RuntimeError(
        f"Konfigurasi berubah tetapi {RESULT_FILE.name} sudah berisi hasil lama. "
        "Arsipkan atau ubah nama hasil lama sebelum menjalankan konfigurasi baru."
    )

results = list(previous)
completed = {(r["model"], r["row_key"]) for r in results}
print("Fingerprint:", RUN_FINGERPRINT)
print("Hasil yang sudah tersedia:", len(completed))
print("Target total:", len(MODELS) * len(selected))

for model in MODELS:
    print("\nMODEL:", model)
    for ix, record in enumerate(selected, 1):
        key = (model, record["row_key"])
        if key in completed:
            print(f"  {ix}/{len(selected)} {record['row_key']}: sudah tercatat")
            continue

        endpoint, payload = build_request(model, record)
        entry = {
            "fingerprint": RUN_FINGERPRINT, "model": model,
            "row_key": record["row_key"], "source_record_id": record["source_record_id"],
            "language_code": record["language_code"], "group_id": record["group_id"],
            "split": record["split"], "target_function": target_name(record),
            "question": record["question"], "prediction_name": "", "prediction_arguments_json": "",
            "gt_json": compact_json(record["gt"]), "status": "", "parsed_json": False,
            "function_correct": False, "full_match": False,
            "done_reason": "", "prompt_eval_count": "", "eval_count": "",
            "latency_seconds": "", "raw_output": "", "error": "",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
        started = time.perf_counter()
        try:
            response = ollama_request(endpoint, payload)
            entry["latency_seconds"] = round(time.perf_counter() - started, 2)
            raw = (response.get("response", "") if endpoint == "/api/generate"
                   else (response.get("message") or {}).get("content", ""))
            entry["raw_output"] = raw or ""
            entry["done_reason"] = response.get("done_reason", "")
            entry["prompt_eval_count"] = response.get("prompt_eval_count", "")
            entry["eval_count"] = response.get("eval_count", "")
            prediction, parsing = parse_model_json(raw)
            status, fn_ok, full_ok = score_internal(prediction, parsing, record["gt"])
            if response.get("done_reason") == "length" and status != "correct":
                status = "output_truncated"
            entry.update({"status": status, "parsed_json": parsing == "parsed",
                          "function_correct": fn_ok, "full_match": full_ok})
            if prediction:
                entry["prediction_name"] = prediction["name"]
                entry["prediction_arguments_json"] = compact_json(prediction["arguments"])
        except Exception as exc:
            entry["latency_seconds"] = round(time.perf_counter() - started, 2)
            entry["status"] = "api_error"
            entry["error"] = str(exc)[:1000]

        results.append(entry)
        completed.add(key)
        save_predictions(results)
        print(f"  {ix}/{len(selected)} {record['row_key']}: {entry['status']} ({entry['latency_seconds']} s)")
        if entry["error"]:
            print("    Error:", entry["error"][:250])
        if isinstance(entry["prompt_eval_count"], int) and entry["prompt_eval_count"] >= OPTIONS["num_ctx"] - 384:
            print("    PERINGATAN: prompt mendekati batas num_ctx; periksa kemungkinan pemotongan konteks.")

    # Lepaskan model setelah pengujian selesai.
    try:
        ollama_request("/api/generate", {"model": model, "keep_alive": 0}, timeout=60)
    except Exception:
        pass

CONFIG_FILE.write_text(json.dumps({
    "mode": RUN_MODE, "fingerprint": RUN_FINGERPRINT,
    "model_digests": {m: installed[m].get("digest") for m in MODELS},
    "models": MODELS, "seed": RANDOM_SEED, "options": OPTIONS,
    "prompt_version": PROMPT_VERSION,
    "records": len(selected), "request_total": len(selected) * len(MODELS),
    "dataset_summary": review_summary,
    "scoring": "Internal accepted-alternative exact match; not official BFCL AST Accuracy",
    "model_transport": {m: ("raw_gemma2_prompt" if m.startswith("Supa-AI/") else "chat_json") for m in MODELS},
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nTersimpan:", RESULT_FILE)
print("Konfigurasi:", CONFIG_FILE)

Fingerprint: 7b43396edd47f0f7
Hasil yang sudah tersedia: 0
Target total: 12

MODEL: qwen2.5:7b-instruct
  1/4 en-US:0000529: correct (31.29 s)
  2/4 id-ID:0000361: correct (0.59 s)
  3/4 en-US:0001800: wrong_function (0.42 s)
  4/4 id-ID:0001211: wrong_function (0.37 s)

MODEL: gemma2:9b-instruct-q4_K_M
  1/4 en-US:0000529: invalid_structure (20.68 s)
  2/4 id-ID:0000361: invalid_structure (3.04 s)
  3/4 en-US:0001800: invalid_structure (15.03 s)
  4/4 id-ID:0001211: invalid_structure (18.26 s)

MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
  1/4 en-US:0000529: invalid_structure (21.22 s)
  2/4 id-ID:0000361: invalid_structure (29.77 s)
  3/4 en-US:0001800: invalid_structure (25.77 s)
  4/4 id-ID:0001211: invalid_structure (28.72 s)

Tersimpan: D:\KP-Agentic-AI\results\baseline_evaluation\predictions_pilot.csv
Konfigurasi: D:\KP-Agentic-AI\results\baseline_evaluation\config_pilot.json


## 9. Ringkasan hasil

Hasil diringkas berdasarkan model dan bahasa, mencakup validitas format, ketepatan fungsi, kecocokan fungsi dan argumen, serta jenis kesalahan.

Pilot hanya digunakan untuk memeriksa apakah alur evaluasi berjalan. Jumlah sampelnya belum cukup untuk menentukan model dengan performa terbaik.

In [9]:
def as_bool(value):
    return value is True or str(value).lower() == "true"

status_values = ["correct", "wrong_function", "wrong_argument_keys", "wrong_argument_values",
                 "invalid_json", "invalid_structure", "output_truncated", "api_error"]
summary_rows = []

for model in MODELS:
    for language in ("en-US", "id-ID"):
        subset = [r for r in results if r["model"] == model and r["language_code"] == language]
        if not subset:
            continue
        total = len(subset)
        statuses = Counter(r["status"] for r in subset)
        fn = sum(as_bool(r["function_correct"]) for r in subset)
        full = sum(as_bool(r["full_match"]) for r in subset)
        parsed_count = sum(as_bool(r["parsed_json"]) for r in subset)
        item = {
            "model": model, "language_code": language, "total": total,
            "format_valid_rate": round(parsed_count / total, 4),
            "function_accuracy": round(fn / total, 4),
            "full_match_rate_internal": round(full / total, 4),
            **{f"n_{status}": statuses.get(status, 0) for status in status_values},
        }
        summary_rows.append(item)

with SUMMARY_FILE.open("w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except ImportError:
    for item in summary_rows:
        print(item)

print("\nTotal hasil:", len(results))
print("Ringkasan:", SUMMARY_FILE)
print("Status semua hasil:", dict(Counter(r["status"] for r in results)))

errors = [r for r in results if r["status"] != "correct"]
if errors:
    print("\nContoh kesalahan (maksimal 6):")
    for r in errors[:6]:
        print(r["model"], r["row_key"], r["status"], "pred:", r["prediction_name"])
else:
    print("Semua prediksi pada data yang diuji cocok menurut scorer internal.")

,model,language_code,total,format_valid_rate,function_accuracy,full_match_rate_internal,n_correct,n_wrong_function,n_wrong_argument_keys,n_wrong_argument_values,n_invalid_json,n_invalid_structure,n_output_truncated,n_api_error
0,qwen2.5:7b-instruct,en-US,2,1.0,0.5,0.5,1,1,0,0,0,0,0,0
1,qwen2.5:7b-instruct,id-ID,2,1.0,0.5,0.5,1,1,0,0,0,0,0,0
2,gemma2:9b-instruct-q4_K_M,en-US,2,0.0,0.0,0.0,0,0,0,0,0,2,0,0
3,gemma2:9b-instruct-q4_K_M,id-ID,2,0.0,0.0,0.0,0,0,0,0,0,2,0,0
4,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,en-US,2,0.0,0.0,0.0,0,0,0,0,0,2,0,0
5,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,id-ID,2,0.0,0.0,0.0,0,0,0,0,0,2,0,0



Total hasil: 12
Ringkasan: D:\KP-Agentic-AI\results\baseline_evaluation\summary_pilot.csv
Status semua hasil: {'correct': 2, 'wrong_function': 2, 'invalid_structure': 8}

Contoh kesalahan (maksimal 6):
qwen2.5:7b-instruct en-US:0001800 wrong_function pred: social.query
qwen2.5:7b-instruct id-ID:0001211 wrong_function pred: lists.query
gemma2:9b-instruct-q4_K_M en-US:0000529 invalid_structure pred: 
gemma2:9b-instruct-q4_K_M id-ID:0000361 invalid_structure pred: 
gemma2:9b-instruct-q4_K_M en-US:0001800 invalid_structure pred: 
gemma2:9b-instruct-q4_K_M id-ID:0001211 invalid_structure pred: 


In [10]:

# Pilot tambahan menggunakan katalog 55 fungsi yang lebih ringkas

def compact_catalog(record):
    lines = []

    for func in record["function"]:
        description = func.get("description", "")
        description = description.split("The default user experience")[0]
        description = description.split(". ")[0].strip()[:100]

        properties = func.get("parameters", {}).get("properties", {})
        arguments = ", ".join(properties.keys())

        lines.append(
            f"{func['name']}: {description}\n"
            f"  Arguments: {arguments if arguments else 'none'}"
        )

    return "\n".join(lines)


def compact_prompt(record):
    return (
        "Select the most appropriate function for the user request.\n"
        "Return one JSON object with exactly two keys: name and arguments.\n"
        "The name must be a function from the catalog.\n"
        "Arguments must contain the relevant parameter names and string values.\n"
        "Do not copy function definitions or include explanations.\n\n"
        "Available functions:\n"
        + compact_catalog(record)
        + "\n\nUser request:\n"
        + record["question"]
        + '\n\nJSON output: {"name":"function_name","arguments":{}}'
    )


compact_results = []

print("Jumlah fungsi:", len(selected[0]["function"]))
print("Panjang prompt baru:", len(compact_prompt(selected[0])), "karakter")

for model in MODELS:
    print("\nMODEL:", model)

    for record in selected:
        content = compact_prompt(record)

        payload = {
            "model": model,
            "stream": False,
            "options": dict(OPTIONS),
            "keep_alive": "5m"
        }

        if model.startswith("Supa-AI/"):
            endpoint = "/api/generate"
            payload["prompt"] = (
                "<start_of_turn>user\n"
                + content
                + "\n<end_of_turn>\n"
                + "<start_of_turn>model\n"
            )
            payload["raw"] = True
            payload["options"]["stop"] = ["<end_of_turn>", "<eos>"]
        else:
            endpoint = "/api/chat"
            payload["messages"] = [{"role": "user", "content": content}]
            payload["format"] = "json"

        try:
            response = ollama_request(endpoint, payload)

            raw = (
                response.get("response", "")
                if endpoint == "/api/generate"
                else response.get("message", {}).get("content", "")
            )

            prediction, parsing = parse_model_json(raw)
            status = score_internal(
                prediction, parsing, record["gt"]
            )[0]

            compact_results.append({
                "model": model,
                "record_id": record["source_record_id"],
                "language": record["language_code"],
                "status": status,
                "raw_output": raw,
                "prompt_tokens": response.get("prompt_eval_count")
            })

            print(
                record["language_code"],
                record["source_record_id"],
                "|", status,
                "| tokens:", response.get("prompt_eval_count")
            )
            print("Output:", repr(raw[:350]))

        except Exception as e:
            print("ERROR:", str(e))
            compact_results.append({
                "model": model,
                "record_id": record["source_record_id"],
                "language": record["language_code"],
                "status": "api_error",
                "raw_output": str(e)
            })

print("\n=== RINGKASAN PILOT KATALOG RINGKAS ===")

for model in MODELS:
    rows = [r for r in compact_results if r["model"] == model]
    print(model)
    print("Benar:", sum(r["status"] == "correct" for r in rows), "/", len(rows))
    print("Status:", dict(Counter(r["status"] for r in rows)))


Jumlah fungsi: 55
Panjang prompt baru: 6999 karakter

MODEL: qwen2.5:7b-instruct
en-US 4168 | correct | tokens: 1581
Output: '{"name":"weather.query","arguments":{"place_name":"Kansas"}}'
id-ID 4168 | correct | tokens: 1583
Output: '{"name":"weather.query","arguments":{"place_name":"kendari"}}'
en-US 15794 | wrong_function | tokens: 1578
Output: '{"name":"recommendation.locations","arguments":{"business_name":"Seth","business_type":"residence"}}'
id-ID 15794 | wrong_function | tokens: 1580
Output: '{"name":"qa.definition","arguments":{"definition_word":"siska"}}'

MODEL: gemma2:9b-instruct-q4_K_M
en-US 4168 | correct | tokens: 1733
Output: '{"name": "weather.query", "arguments": {"place_name": "kansas"}}'
id-ID 4168 | correct | tokens: 1733
Output: '{"name": "weather.query", "arguments": {"place_name": "kendari"}}'
en-US 15794 | wrong_function | tokens: 1730
Output: '{"name": "recommendation.locations", "arguments": {"business_name": "seth", "business_type": null, "food_type": null, "p

In [11]:

# Pemeriksaan record 15794

import json

for record in selected:
    if str(record["source_record_id"]) != "15794":
        continue

    print("=" * 60)
    print("ID:", record["source_record_id"])
    print("Bahasa:", record["language_code"])
    print("Pertanyaan:", record["question"])

    print("\nGROUND TRUTH:")
    print(json.dumps(record["gt"], indent=2, ensure_ascii=False))

    print("\nDEFINISI FUNGSI TERKAIT:")

    related = {
        "recommendation.locations",
        "qa.definition",
        "qa.factoid"
    }

    gt = record["gt"]
    if isinstance(gt, dict):
        related.update(gt.keys())
        if isinstance(gt.get("name"), str):
            related.add(gt["name"])

    for func in record["function"]:
        if func["name"] in related:
            print("\nNama:", func["name"])
            print("Deskripsi:", func.get("description", ""))
            print("Parameter:",
                  list(func.get("parameters", {})
                       .get("properties", {}).keys()))


ID: 15794
Bahasa: en-US
Pertanyaan: where seth lives

GROUND TRUTH:
{
  "email.querycontact": {
    "person": [
      "seth"
    ],
    "personal_info": [
      "lives"
    ],
    "business_name": [
      ""
    ],
    "relation": [
      ""
    ]
  }
}

DEFINISI FUNGSI TERKAIT:

Nama: email.querycontact
Deskripsi: Query information for an email contact. The API cannot be called without any arguments. The default user experience is that it opens the email app to the contact info for requested contact and reads out requested information
Parameter: ['business_name', 'person', 'personal_info', 'relation']

Nama: qa.definition
Deskripsi: Ask for a definition of a given word. The API cannot be called without any arguments. The default user experience is that it responds to the user on screen with an answer and links to the rag sources used
Parameter: ['definition_word']

Nama: qa.factoid
Deskripsi: Ask a factual question about historical figures, people, places, movies and so on. The API ca

In [12]:

# Pilot lanjutan pada tiga pasangan development

import random
from collections import Counter

index = {
    lang: {str(r["source_record_id"]): r for r in rows}
    for lang, rows in dataset.items()
}

used_ids = {str(r["source_record_id"]) for r in selected}
used_functions = {target_name(r) for r in selected}

ids = list(set(index["en-US"]) & set(index["id-ID"]))
random.Random(20261020).shuffle(ids)

new_records = []

for rid in ids:
    if rid in used_ids:
        continue

    en = index["en-US"][rid]
    ind = index["id-ID"][rid]
    fn = target_name(en)

    if fn != target_name(ind) or fn in used_functions:
        continue

    new_records.extend([en, ind])
    used_functions.add(fn)

    if len(new_records) == 6:
        break

assert len(new_records) == 6, "Pasangan tidak mencukupi"

pilot_lanjutan = []

for model in MODELS:
    print("\nMODEL:", model)

    for record in new_records:
        content = compact_prompt(record)

        payload = {
            "model": model,
            "stream": False,
            "options": dict(OPTIONS),
            "keep_alive": "5m"
        }

        if model.startswith("Supa-AI/"):
            endpoint = "/api/generate"
            payload["prompt"] = (
                "<start_of_turn>user\n"
                + content
                + "\n<end_of_turn>\n"
                + "<start_of_turn>model\n"
            )
            payload["raw"] = True
            payload["options"]["stop"] = [
                "<end_of_turn>", "<eos>"
            ]
        else:
            endpoint = "/api/chat"
            payload["messages"] = [
                {"role": "user", "content": content}
            ]
            payload["format"] = "json"

        try:
            response = ollama_request(endpoint, payload)

            raw = (
                response.get("response", "")
                if endpoint == "/api/generate"
                else response.get("message", {}).get("content", "")
            )

            prediction, parsing = parse_model_json(raw)
            status = score_internal(
                prediction, parsing, record["gt"]
            )[0]

        except Exception as e:
            raw = str(e)
            status = "api_error"

        pilot_lanjutan.append({
            "model": model,
            "record_id": record["source_record_id"],
            "language": record["language_code"],
            "target": target_name(record),
            "status": status,
            "output": raw
        })

        print(
            record["source_record_id"],
            record["language_code"],
            "| GT:", target_name(record),
            "|", status
        )
        print("Output:", repr(raw[:250]))

print("\n=== RINGKASAN PILOT LANJUTAN ===")

for model in MODELS:
    rows = [r for r in pilot_lanjutan if r["model"] == model]

    print("\n", model)
    print("Status:", dict(Counter(r["status"] for r in rows)))
    print(
        "Benar:",
        sum(r["status"] == "correct" for r in rows),
        "/", len(rows)
    )



MODEL: qwen2.5:7b-instruct
11476 en-US | GT: play.podcasts | wrong_argument_values
Output: '{"name":"play.podcasts","arguments":{"podcast_descriptor":"previous"}}'
11476 id-ID | GT: play.podcasts | wrong_argument_values
Output: '{"name":"play.podcasts","arguments":{"podcast_descriptor":"previous"}}'
13679 en-US | GT: qa.factoid | wrong_function
Output: '{"name":"qa.definition","arguments":{"definition_word":"stomachs in a cow"}}'
13679 id-ID | GT: qa.factoid | wrong_function
Output: '{"name":"qa.definition","arguments":{"definition_word":"perut sapi"}}'
4361 en-US | GT: play.music | wrong_function
Output: '{"name":"music.likeness","arguments":{"artist_name":"goodsounds"}}'
4361 id-ID | GT: play.music | wrong_argument_values
Output: '{"name":"play.music","arguments":{"music_genre":"bagus"}}'

MODEL: gemma2:9b-instruct-q4_K_M
11476 en-US | GT: play.podcasts | wrong_argument_values
Output: '{"name": "play.podcasts", "arguments": {"person": null, "player_setting": "resume", "podcast_descr

In [13]:

# Pemeriksaan GT pada pilot lanjutan

import json

for record in new_records:
    function_name, arguments = next(iter(record["gt"].items()))

    print("=" * 60)
    print("ID:", record["source_record_id"])
    print("Bahasa:", record["language_code"])
    print("Pertanyaan:", record["question"])
    print("Fungsi GT:", function_name)
    print("Argumen GT:")
    print(json.dumps(arguments, indent=2, ensure_ascii=False))


ID: 11476
Bahasa: en-US
Pertanyaan: play the previous podcast
Fungsi GT: play.podcasts
Argumen GT:
{
  "podcast_descriptor": [
    "previous podcast"
  ],
  "person": [
    ""
  ],
  "player_setting": [
    ""
  ],
  "podcast_name": [
    ""
  ]
}
ID: 11476
Bahasa: id-ID
Pertanyaan: putar podcast sebelumnya
Fungsi GT: play.podcasts
Argumen GT:
{
  "podcast_descriptor": [
    "podcast sebelumnya"
  ],
  "person": [
    ""
  ],
  "player_setting": [
    ""
  ],
  "podcast_name": [
    ""
  ]
}
ID: 13679
Bahasa: en-US
Pertanyaan: can you tell me how many stomachs a cow has
Fungsi GT: qa.factoid
Argumen GT:
{
  "query": [
    "can you tell me how many stomachs a cow has"
  ]
}
ID: 13679
Bahasa: id-ID
Pertanyaan: bisakah anda memberitahu berapa banyak perut yang dimiliki sapi
Fungsi GT: qa.factoid
Argumen GT:
{
  "query": [
    "bisakah anda memberitahu berapa banyak perut yang dimiliki sapi"
  ]
}
ID: 4361
Bahasa: en-US
Pertanyaan: give me some good sounds
Fungsi GT: play.music
Argumen GT:

In [14]:

# Evaluasi prompt ringkas versi 2

def compact_prompt_v2(record):
    catalog = []

    for func in record["function"]:
        description = func.get("description", "")
        description = description.split(
            "The default user experience"
        )[0].strip()

        properties = func.get(
            "parameters", {}
        ).get("properties", {})

        required = set(
            func.get("parameters", {}).get("required", [])
        )

        lines = [f"{func['name']}: {description}"]

        for key, meta in properties.items():
            detail = meta.get("description", "")
            status = "required" if key in required else "optional"
            lines.append(f"  {key} ({status}): {detail}")

        catalog.append("\n".join(lines))

    return (
        "Select exactly one function for the user request.\n"
        "Return only valid JSON with keys 'name' and 'arguments'.\n"
        "Use the exact function and parameter names.\n"
        "Use string values and omit unspecified optional arguments.\n"
        "Do not invent values or add explanations.\n\n"
        "Available functions:\n"
        + "\n".join(catalog)
        + "\n\nUser request:\n"
        + record["question"]
        + "\n\nJSON output:"
    )


records_v2 = selected + new_records
hasil_v2 = []

print("Total record:", len(records_v2))
print("Jumlah fungsi:", len(records_v2[0]["function"]))
print("Panjang prompt:", len(compact_prompt_v2(records_v2[0])))

for model in MODELS:
    print("\nMODEL:", model)

    for record in records_v2:
        content = compact_prompt_v2(record)

        payload = {
            "model": model,
            "stream": False,
            "options": dict(OPTIONS),
            "keep_alive": "5m"
        }

        if model.startswith("Supa-AI/"):
            endpoint = "/api/generate"
            payload["raw"] = True
            payload["prompt"] = (
                "<start_of_turn>user\n"
                + content
                + "\n<end_of_turn>\n"
                + "<start_of_turn>model\n"
            )
            payload["options"]["stop"] = [
                "<end_of_turn>", "<eos>"
            ]
        else:
            endpoint = "/api/chat"
            payload["messages"] = [
                {"role": "user", "content": content}
            ]
            payload["format"] = "json"

        try:
            response = ollama_request(endpoint, payload)

            raw = (
                response.get("response", "")
                if endpoint == "/api/generate"
                else response.get("message", {}).get("content", "")
            )

            prediction, parsing = parse_model_json(raw)
            status, fn_ok, full_ok = score_internal(
                prediction, parsing, record["gt"]
            )

        except Exception as e:
            status = "api_error"
            fn_ok = False
            full_ok = False
            raw = str(e)

        hasil_v2.append({
            "model": model,
            "record_id": record["source_record_id"],
            "language": record["language_code"],
            "status": status,
            "function_correct": fn_ok,
            "full_match": full_ok,
            "raw_output": raw
        })

        print(
            record["source_record_id"],
            record["language_code"],
            "|", status
        )

print("\n=== RINGKASAN PROMPT V2 ===")

for model in MODELS:
    rows = [r for r in hasil_v2 if r["model"] == model]

    print("\n", model)
    print("Fungsi benar:", sum(r["function_correct"] for r in rows),
          "/", len(rows))
    print("Full match:", sum(r["full_match"] for r in rows),
          "/", len(rows))
    print("Jenis hasil:", dict(Counter(r["status"] for r in rows)))


Total record: 10
Jumlah fungsi: 55
Panjang prompt: 20840

MODEL: qwen2.5:7b-instruct
4168 en-US | correct
4168 id-ID | correct
15794 en-US | wrong_function
15794 id-ID | wrong_function
11476 en-US | invalid_structure
11476 id-ID | invalid_structure
13679 en-US | wrong_function
13679 id-ID | wrong_function
4361 en-US | wrong_function
4361 id-ID | correct

MODEL: gemma2:9b-instruct-q4_K_M
4168 en-US | correct
4168 id-ID | correct
15794 en-US | wrong_function
15794 id-ID | wrong_function
11476 en-US | wrong_argument_keys
11476 id-ID | wrong_argument_keys
13679 en-US | wrong_argument_values
13679 id-ID | wrong_argument_values
4361 en-US | wrong_function
4361 id-ID | correct

MODEL: Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m
4168 en-US | correct
4168 id-ID | correct
15794 en-US | wrong_function
15794 id-ID | wrong_function
11476 en-US | wrong_argument_keys
11476 id-ID | wrong_argument_keys
13679 en-US | wrong_argument_values
13679 id-ID | wrong_argument_values
4361 en-US | wrong_fun

In [16]:

# Pemeriksaan respons invalid_structure pada Prompt V2

for row in hasil_v2:
    if (
        row["model"] == "qwen2.5:7b-instruct"
        and row["status"] == "invalid_structure"
    ):
        print("=" * 60)
        print("Record ID:", row["record_id"])
        print("Bahasa:", row["language"])
        print("Respons asli:")
        print(row["raw_output"])

        try:
            obj = json.loads(row["raw_output"])
            print("\nTipe JSON:", type(obj).__name__)

            if isinstance(obj, dict):
                print("Key JSON:", list(obj.keys()))
        except json.JSONDecodeError:
            print("\nJSON tidak dapat diparsing")


Record ID: 11476
Bahasa: en-US
Respons asli:
{
  "name": "play.podcasts",
  "player_setting": "previous"
}

Tipe JSON: dict
Key JSON: ['name', 'player_setting']
Record ID: 11476
Bahasa: id-ID
Respons asli:
{
  "name": "play.podcasts",
  "player_setting": "replay"
}

Tipe JSON: dict
Key JSON: ['name', 'player_setting']


In [17]:

# Pemilihan sampel pilot lanjutan

import random

tested_ids = {
    "4168", "15794", "11476", "13679", "4361"
}

index = {
    lang: {
        str(r["source_record_id"]): r
        for r in records
    }
    for lang, records in dataset.items()
}

candidate_ids = sorted(
    set(index["en-US"]) & set(index["id-ID"])
)

random.Random(20261021).shuffle(candidate_ids)

chosen_ids = []
used_functions = set()

for rid in candidate_ids:
    if rid in tested_ids:
        continue

    en = index["en-US"][rid]
    ind = index["id-ID"][rid]

    if target_name(en) != target_name(ind):
        continue

    function_name = target_name(en)

    if function_name in used_functions:
        continue

    chosen_ids.append(rid)
    used_functions.add(function_name)

    if len(chosen_ids) == 10:
        break

assert len(chosen_ids) == 10, "Sampel tidak mencukupi"

pilot_baru = [
    index[lang][rid]
    for rid in chosen_ids
    for lang in ("en-US", "id-ID")
]

print("Jumlah pasangan:", len(chosen_ids))
print("Jumlah record:", len(pilot_baru))
print("Jumlah fungsi berbeda:", len(used_functions))

for rid in chosen_ids:
    print(rid, "|", target_name(index["en-US"][rid]))


Jumlah pasangan: 10
Jumlah record: 20
Jumlah fungsi berbeda: 10
10093 | cooking.recipe
11598 | recommendation.events
1913 | alarm.set
13907 | qa.definition
1803 | news.query
2263 | iot.hue_lightdim
3809 | audio.volume_down
14601 | qa.factoid
5243 | iot.hue_lightoff
2206 | audio.volume_mute


In [18]:

# Pilot V1 pada 20 record development

import csv
import hashlib
import json
from collections import Counter
from datetime import datetime, timezone

assert len(pilot_baru) == 20
assert len({r["row_key"] for r in pilot_baru}) == 20
assert all(r["split"] == "development" for r in pilot_baru)
assert len({target_name(r) for r in pilot_baru}) == 10

pilot_file = OUT / "pilot_20_v1.csv"
config_file = OUT / "pilot_20_v1_config.json"

prompt_hash = hashlib.sha256(
    json.dumps(
        [compact_prompt(r) for r in pilot_baru],
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

config = {
    "experiment": "pilot_20_v1",
    "prompt_version": "compact_catalog_v1",
    "prompt_hash": prompt_hash,
    "models": MODELS,
    "model_digests": {
        m: installed[m]["digest"] for m in MODELS
    },
    "options": OPTIONS,
    "record_keys": [r["row_key"] for r in pilot_baru],
    "dataset_split": "development",
    "scoring": "internal_exact_match",
    "transports": {
        m: (
            "raw_gemma2_prompt"
            if m.startswith("Supa-AI/")
            else "chat_json"
        )
        for m in MODELS
    }
}

fingerprint = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode("utf-8")
).hexdigest()

config["fingerprint"] = fingerprint

if config_file.exists():
    old_config = json.loads(
        config_file.read_text(encoding="utf-8")
    )
    if old_config["fingerprint"] != fingerprint:
        raise RuntimeError(
            "Konfigurasi berbeda dari pengujian tersimpan."
        )
elif pilot_file.exists():
    raise RuntimeError(
        "CSV sudah ada tetapi konfigurasi tidak ditemukan."
    )
else:
    config_file.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

fields = [
    "model", "row_key", "record_id", "language",
    "target_function", "predicted_function",
    "status", "function_correct", "full_match",
    "parsed_json", "prompt_tokens", "output_tokens",
    "done_reason", "raw_output", "error", "timestamp_utc"
]

existing = []

if pilot_file.exists():
    with pilot_file.open(
        "r", encoding="utf-8-sig", newline=""
    ) as f:
        existing = list(csv.DictReader(f))

done = {
    (r["model"], r["row_key"]) for r in existing
}

print("Total target:", len(pilot_baru) * len(MODELS))
print("Sudah tersimpan:", len(done))
print("Sisa:", 60 - len(done))

for model in MODELS:
    print("\nMODEL:", model)

    for record in pilot_baru:
        key = (model, record["row_key"])

        if key in done:
            continue

        content = compact_prompt(record)

        payload = {
            "model": model,
            "stream": False,
            "options": dict(OPTIONS),
            "keep_alive": "5m"
        }

        if model.startswith("Supa-AI/"):
            endpoint = "/api/generate"
            payload["raw"] = True
            payload["prompt"] = (
                "<start_of_turn>user\n"
                + content
                + "\n<end_of_turn>\n"
                + "<start_of_turn>model\n"
            )
            payload["options"]["stop"] = [
                "<end_of_turn>", "<eos>"
            ]
        else:
            endpoint = "/api/chat"
            payload["messages"] = [
                {"role": "user", "content": content}
            ]
            payload["format"] = "json"

        raw = ""
        error = ""
        prediction = None
        parsed = "invalid_json"
        status = "api_error"
        fn_ok = False
        full_ok = False
        response = {}

        try:
            response = ollama_request(endpoint, payload)

            if endpoint == "/api/generate":
                raw = response.get("response", "")
            else:
                raw = response.get(
                    "message", {}
                ).get("content", "")

            prediction, parsed = parse_model_json(raw)

            if response.get("done_reason") == "length":
                status = "output_truncated"
            else:
                status, fn_ok, full_ok = score_internal(
                    prediction, parsed, record["gt"]
                )

        except Exception as e:
            error = str(e)
            status = "api_error"

        result = {
            "model": model,
            "row_key": record["row_key"],
            "record_id": record["source_record_id"],
            "language": record["language_code"],
            "target_function": target_name(record),
            "predicted_function": (
                prediction["name"]
                if isinstance(prediction, dict)
                else ""
            ),
            "status": status,
            "function_correct": fn_ok,
            "full_match": full_ok,
            "parsed_json": parsed == "parsed",
            "prompt_tokens": response.get("prompt_eval_count"),
            "output_tokens": response.get("eval_count"),
            "done_reason": response.get("done_reason", ""),
            "raw_output": raw,
            "error": error,
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        with pilot_file.open(
            "a", encoding="utf-8-sig", newline=""
        ) as f:
            writer = csv.DictWriter(
                f, fieldnames=fields
            )
            if f.tell() == 0:
                writer.writeheader()
            writer.writerow(result)

        done.add(key)

        print(
            record["source_record_id"],
            record["language_code"],
            "|", status
        )

# Ringkasan pilot
with pilot_file.open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    pilot20_results = list(csv.DictReader(f))

print("\n=== RINGKASAN PILOT 20 RECORD ===")

def is_true(value):
    return str(value).lower() == "true"

for model in MODELS:
    rows = [
        r for r in pilot20_results
        if r["model"] == model
    ]

    total = len(rows)
    fn = sum(is_true(r["function_correct"]) for r in rows)
    full = sum(is_true(r["full_match"]) for r in rows)
    valid = sum(is_true(r["parsed_json"]) for r in rows)

    print("\n", model)
    print("Jumlah:", total)
    print("Format valid:", valid, "/", total)
    print("Fungsi benar:", fn, "/", total)
    print("Full match:", full, "/", total)
    print("Status:", dict(Counter(
        r["status"] for r in rows
    )))

print("\nHasil tersimpan:", pilot_file)
print("Konfigurasi:", config_file)


Total target: 60
Sudah tersimpan: 0
Sisa: 60

MODEL: qwen2.5:7b-instruct
10093 en-US | correct
10093 id-ID | correct
11598 en-US | wrong_argument_values
11598 id-ID | wrong_argument_values
1913 en-US | wrong_argument_values
1913 id-ID | wrong_function
13907 en-US | correct
13907 id-ID | correct
1803 en-US | wrong_function
1803 id-ID | wrong_function
2263 en-US | wrong_argument_values
2263 id-ID | wrong_argument_values
3809 en-US | correct
3809 id-ID | correct
14601 en-US | wrong_argument_values
14601 id-ID | correct
5243 en-US | correct
5243 id-ID | correct
2206 en-US | correct
2206 id-ID | wrong_function

MODEL: gemma2:9b-instruct-q4_K_M
10093 en-US | wrong_argument_values
10093 id-ID | wrong_argument_values
11598 en-US | wrong_argument_keys
11598 id-ID | wrong_argument_keys
1913 en-US | wrong_argument_values
1913 id-ID | wrong_argument_values
13907 en-US | correct
13907 id-ID | correct
1803 en-US | wrong_argument_values
1803 id-ID | wrong_function
2263 en-US | wrong_argument_values
2

In [19]:
# Konfigurasi baseline development

RUN_MODE = "development_full"
PROMPT_VERSION = "compact_catalog_v1_frozen"
KEEP_ALIVE = "5m"

# Menggunakan prompt V1 yang telah diuji
task_content = compact_prompt

# Seluruh data development
selected = sorted(
    all_rows,
    key=lambda r: (
        str(r["source_record_id"]),
        r["language_code"]
    )
)

assert len(selected) == 979
assert len({r["row_key"] for r in selected}) == 979
assert all(r["split"] == "development" for r in selected)
assert all(len(r["function"]) == 55 for r in selected)

print("Mode:", RUN_MODE)
print("Prompt:", PROMPT_VERSION)
print("Jumlah record:", len(selected))
print("Jumlah model:", len(MODELS))
print("Total pengujian:", len(selected) * len(MODELS))
print("Panjang contoh prompt:", len(task_content(selected[0])))

Mode: development_full
Prompt: compact_catalog_v1_frozen
Jumlah record: 979
Jumlah model: 3
Total pengujian: 2937
Panjang contoh prompt: 6994


In [20]:

# Baseline Evaluation - Development Set

import csv
import hashlib
import json
import time
from collections import Counter
from datetime import datetime, timezone

# False: tidak menjalankan inferensi
# True: menjalankan evaluasi development
RUN_DEVELOPMENT = False

def prompt_baseline_v1(record):
    catalog = []

    for func in record["function"]:
        description = func.get("description", "")
        description = description.split(
            "The default user experience"
        )[0]
        description = description.split(". ")[0].strip()[:100]

        properties = func.get(
            "parameters", {}
        ).get("properties", {})

        arguments = ", ".join(properties.keys())

        catalog.append(
            f"{func['name']}: {description}\n"
            f"  Arguments: {arguments if arguments else 'none'}"
        )

    return (
        "Select the most appropriate function for the user request.\n"
        "Return one JSON object with exactly two keys: name and arguments.\n"
        "The name must be a function from the catalog.\n"
        "Arguments must contain the relevant parameter names and string values.\n"
        "Do not copy function definitions or include explanations.\n\n"
        "Available functions:\n"
        + "\n".join(catalog)
        + "\n\nUser request:\n"
        + record["question"]
        + '\n\nJSON output: {"name":"function_name","arguments":{}}'
    )


development_records = sorted(
    all_rows,
    key=lambda r: (
        str(r["source_record_id"]),
        r["language_code"]
    )
)

assert len(development_records) == 979
assert len({r["row_key"] for r in development_records}) == 979
assert all(r["split"] == "development" for r in development_records)
assert all(len(r["function"]) == 55 for r in development_records)

output_file = OUT / "predictions_development_v1.csv"
summary_file = OUT / "summary_development_v1.csv"
config_file = OUT / "config_development_v1.json"

# Identitas konfigurasi eksperimen
prompt_hash = hashlib.sha256()

for record in development_records:
    prompt_hash.update(
        (
            record["row_key"]
            + "\n"
            + prompt_baseline_v1(record)
            + "\n"
        ).encode("utf-8")
    )

config = {
    "mode": "development_full",
    "prompt_version": "compact_catalog_v1",
    "prompt_hash": prompt_hash.hexdigest(),
    "models": MODELS,
    "model_digests": {
        model: installed[model]["digest"]
        for model in MODELS
    },
    "options": OPTIONS,
    "record_keys": [
        record["row_key"]
        for record in development_records
    ],
    "scoring": "internal_exact_match"
}

fingerprint = hashlib.sha256(
    json.dumps(
        config, sort_keys=True
    ).encode("utf-8")
).hexdigest()

config["fingerprint"] = fingerprint

fields = [
    "fingerprint", "model", "row_key", "record_id",
    "language", "target_function", "predicted_function",
    "status", "function_correct", "full_match",
    "parsed_json", "prompt_tokens", "output_tokens",
    "done_reason", "raw_output", "error", "timestamp_utc"
]


def save_development(rows):
    temp_file = output_file.with_suffix(".tmp")

    with temp_file.open(
        "w", encoding="utf-8-sig", newline=""
    ) as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

    temp_file.replace(output_file)


print("=== KONFIGURASI DEVELOPMENT ===")
print("Record:", len(development_records))
print("Model:", len(MODELS))
print("Total pengujian:", len(development_records) * len(MODELS))
print("Jumlah fungsi:", len(development_records[0]["function"]))
print("Prompt:", config["prompt_version"])
print("Mode aktif:", RUN_DEVELOPMENT)

if RUN_DEVELOPMENT:

    if config_file.exists():
        previous_config = json.loads(
            config_file.read_text(encoding="utf-8")
        )
        if previous_config.get("fingerprint") != fingerprint:
            raise RuntimeError(
                "Konfigurasi berubah. Hasil lama tidak boleh dicampur."
            )

    elif output_file.exists():
        raise RuntimeError(
            "CSV ditemukan tanpa file konfigurasi."
        )

    else:
        config_file.write_text(
            json.dumps(config, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    results_dev = []

    if output_file.exists():
        with output_file.open(
            "r", encoding="utf-8-sig", newline=""
        ) as f:
            results_dev = list(csv.DictReader(f))

    assert all(
        row["fingerprint"] == fingerprint
        for row in results_dev
    )

    completed = {
        (row["model"], row["row_key"])
        for row in results_dev
    }

    assert len(completed) == len(results_dev), "Hasil duplikat"

    print("\nHasil tersimpan:", len(completed))
    print("Sisa:", 2937 - len(completed))

    for model in MODELS:
        print("\nMODEL:", model)

        for index, record in enumerate(development_records, 1):
            key = (model, record["row_key"])

            if key in completed:
                continue

            content = prompt_baseline_v1(record)

            payload = {
                "model": model,
                "stream": False,
                "options": dict(OPTIONS),
                "keep_alive": "5m"
            }

            if model.startswith("Supa-AI/"):
                endpoint = "/api/generate"
                payload["raw"] = True
                payload["prompt"] = (
                    "<start_of_turn>user\n"
                    + content
                    + "\n<end_of_turn>\n"
                    + "<start_of_turn>model\n"
                )
                payload["options"]["stop"] = [
                    "<end_of_turn>", "<eos>"
                ]
            else:
                endpoint = "/api/chat"
                payload["messages"] = [
                    {"role": "user", "content": content}
                ]
                payload["format"] = "json"

            raw = ""
            error = ""
            prediction = None
            parsing = "invalid_json"
            status = "api_error"
            fn_ok = False
            full_ok = False
            response = {}

            started = time.perf_counter()

            try:
                response = ollama_request(endpoint, payload)

                raw = (
                    response.get("response", "")
                    if endpoint == "/api/generate"
                    else response.get("message", {}).get("content", "")
                )

                prediction, parsing = parse_model_json(raw)

                status, fn_ok, full_ok = score_internal(
                    prediction, parsing, record["gt"]
                )

                if (
                    response.get("done_reason") == "length"
                    and status != "correct"
                ):
                    status = "output_truncated"

            except Exception as exc:
                error = str(exc)
                status = "api_error"

            result = {
                "fingerprint": fingerprint,
                "model": model,
                "row_key": record["row_key"],
                "record_id": record["source_record_id"],
                "language": record["language_code"],
                "target_function": target_name(record),
                "predicted_function": (
                    prediction.get("name", "")
                    if isinstance(prediction, dict)
                    else ""
                ),
                "status": status,
                "function_correct": fn_ok,
                "full_match": full_ok,
                "parsed_json": parsing == "parsed",
                "prompt_tokens": response.get("prompt_eval_count", ""),
                "output_tokens": response.get("eval_count", ""),
                "done_reason": response.get("done_reason", ""),
                "raw_output": raw,
                "error": error,
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            }

            results_dev.append(result)
            completed.add(key)
            save_development(results_dev)

            print(
                index, "/", len(development_records),
                "|", record["language_code"],
                "|", status,
                "|", round(time.perf_counter() - started, 1), "s"
            )

        # Mengosongkan model dari memori setelah selesai
        try:
            ollama_request(
                "/api/generate",
                {"model": model, "keep_alive": 0},
                timeout=60
            )
        except Exception:
            pass

    # Ringkasan evaluasi
    summary_dev = []

    for model in MODELS:
        for language in ("en-US", "id-ID"):
            rows = [
                row for row in results_dev
                if row["model"] == model
                and row["language"] == language
            ]

            total = len(rows)

            if total == 0:
                continue

            function_correct = sum(
                str(row["function_correct"]).lower() == "true"
                for row in rows
            )

            full_match = sum(
                str(row["full_match"]).lower() == "true"
                for row in rows
            )

            valid = sum(
                str(row["parsed_json"]).lower() == "true"
                for row in rows
            )

            summary_dev.append({
                "model": model,
                "language": language,
                "total": total,
                "format_valid": valid,
                "function_correct": function_correct,
                "full_match": full_match,
                "function_accuracy": round(
                    function_correct / total, 4
                ),
                "full_match_rate": round(
                    full_match / total, 4
                )
            })

    with summary_file.open(
        "w", encoding="utf-8-sig", newline=""
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=list(summary_dev[0])
        )
        writer.writeheader()
        writer.writerows(summary_dev)

    print("\n=== HASIL BASELINE DEVELOPMENT ===")

    try:
        import pandas as pd
        display(pd.DataFrame(summary_dev))
    except ImportError:
        for row in summary_dev:
            print(row)

    print("\nHasil:", output_file)
    print("Ringkasan:", summary_file)

else:
    print("\nEvaluasi development belum dijalankan.")
    print("Ubah RUN_DEVELOPMENT = True untuk menjalankannya.")


=== KONFIGURASI DEVELOPMENT ===
Record: 979
Model: 3
Total pengujian: 2937
Jumlah fungsi: 55
Prompt: compact_catalog_v1
Mode aktif: False

Evaluasi development belum dijalankan.
Ubah RUN_DEVELOPMENT = True untuk menjalankannya.
